### **Phase Transition: Number of Neighbors**
This plot illustrates the **phase transition** process as a result of running bash script, multiple run of MD with different target temperatures, by showing the number of neighbors during:
- **Cooling** ( ← blue line)
- **Heating** ( → red line)  

note: The plot uses smoothed data for clarity.  

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter

address = "/home/hadis/selfassembly/buildParticleOriented/buildVS/jan7_dotLong_COMang_diffSeed2"

def load_data(address, process):
    if process == "cooling":
        subfolder = "B_NeighborsCount.dat"
    elif process == "heating":
        subfolder = "B_NeighborsCount2.dat"

    data = np.loadtxt(os.path.join(address, subfolder))
    if process == "heating":
        data = np.flipud(data)

    temp = data[:, 0]
    neighbor_count = data[:, 4]

    return temp, neighbor_count

def smooth_data(temp, neighbor_count, method):
    if method == "savgol":
        smoothed_neighbors = savgol_filter(neighbor_count, window_length=7, polyorder=2)
        smoothed_temp = temp
    elif method == "gaussian":
        smoothed_neighbors = gaussian_filter(neighbor_count, sigma=1)
        smoothed_temp = temp
    else:
        smoothed_temp = temp
        smoothed_neighbors = neighbor_count

    return smoothed_temp, smoothed_neighbors

def plot_phase_trans(address, method, show_original=False):
    cool_data = load_data(address, "cooling")
    heat_data = load_data(address, "heating")
    
    cool_smooth = smooth_data(cool_data[0], cool_data[1], method)
    heat_smooth = smooth_data(heat_data[0], heat_data[1], method)
    
    fig, ax = plt.subplots(figsize=(6,4))
    ax.plot(cool_smooth[0], cool_smooth[1], label=' ← Cool down', color='blue')
    ax.plot(heat_smooth[0], heat_smooth[1], label=' → Heat up', color='red')
    if show_original:
        ax.plot(cool_data[0], cool_data[1], label=' ← Cool down (Original)', color='blue', alpha=0.3)
        ax.plot(heat_data[0], heat_data[1], label=' → Heat up (Original)', color='red', alpha=0.3)

    ax.set_title('Phase Transition - Number of Neighbors', fontsize=12, fontweight='semibold')
    ax.set_xlabel('Temperature', fontweight='semibold')
    ax.set_ylabel('Number of Neighbors', fontweight='semibold')

    ax.grid(visible=True, linestyle='--', linewidth=0.7)
    ax.legend(title="Processes", fontsize=8, title_fontsize=10)

    plt.show()

plot_phase_trans(address, "gaussian")


### **Snap shot of particles configurations**
This plot illustrates the **location** and **orientation** of particles during the cooling down or heating up process, as a result of running bash script, multiple run of MD with different target temperatures, at the desired temperature:
- **Colors show the orientation of each particle**  
---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

def plotSnapShot(directory, start, end):
    items = os.listdir(directory)
    configs = sorted(items)
    temperature = [name.split('_')[-1].replace('.dat', '') for name in configs]

    for i in range(start, end):
        file_path = os.path.join(directory, configs[i])
        res = np.loadtxt(file_path)
        print(res.shape)

        plt.figure(figsize=(6, 4))
        
        scatter = plt.scatter(res[:, 0], res[:, 1], c=res[:, 2], cmap='twilight', s=50, alpha=0.8, vmin=0, vmax=360)
        
        cbar = plt.colorbar(scatter)
        cbar.set_label('φ in degrees')
        
        plt.gca().set_aspect('equal', adjustable='box')
        plt.xlim(0, 20)
        plt.ylim(0, 20)
        plt.xlabel('X Position')
        plt.ylabel('Y Position')
        plt.title(f'Positions + Orientation φ  (T={temperature[i]})')
        plt.grid(True)

        plt.show()

file1 = "/home/hadis/selfassembly/buildParticleOriented/buildVS/nov18/cooling_configs/"

plotSnapShot(file1, 0, 5)